# Step 2 - Baselines From Step1 Outputs

This notebook is the improved baseline pipeline for the expanded 4-action scope.

## Uses Step 1 outputs directly
- `metadata_rgb_multi4_with_split.csv`
- `split_manifest.json`
- `quality_check_summary.json` (optional reference)

## Tasks covered
1. Action classification (4-class)
2. Skill classification (binary: beginner vs expert)
3. Per-action skill metrics (professor confound concern)

## Models
- SVM baseline
- LSTM baseline

## Required principle
- **No re-splitting**. Always use split from Step 1 metadata.


## Clarifications
1. Skill imbalance is handled:
   - SVM: `class_weight='balanced'`
   - LSTM: weighted cross-entropy
2. `repeat_id` is trial number, not skill.
3. Use `action_folder` / `canonical_action` columns for action logic (not raw token alone).
4. Subject-disjoint assurance comes from Step 1 split manifest and leakage report.


In [1]:
!pip -q install ultralytics scikit-learn pandas numpy opencv-python tqdm

import os
import re
import csv
import json
import random
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print('Environment ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.2 MB/s eta 0:00:00
Environment ready.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =============================
# Config
# =============================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

USE_GITHUB_DOWNLOAD = True
EXTRACT_KEYPOINTS = True

ACTION_FOLDERS = ['backhand', 'forehand_flat', 'kick_service', 'smash']

DATA_ROOT = Path('/content/tennis_data')
VIDEO_ROOT = DATA_ROOT / 'videos'
KPT_ROOT = DATA_ROOT / 'keypoints'
RESULT_ROOT = DATA_ROOT / 'results_step2'

for d in [VIDEO_ROOT, KPT_ROOT, RESULT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

# Step 1 output paths (upload to /content first if needed)
STEP1_DIR = Path('/content/drive/MyDrive/step1_outputs')
STEP1_META = STEP1_DIR / 'metadata_rgb_multi4_with_split.csv'
STEP1_MANIFEST = STEP1_DIR / 'split_manifest.json'

POSE_MODEL = 'yolo11n-pose.pt'
CONF_THRESHOLD = 0.15
MAX_FRAMES = None

# Optional slower but more report-rigorous setting
RUN_MULTI_SEED = True
MULTI_SEEDS = [13, 42, 77]

print('STEP1_META exists:', STEP1_META.exists())
print('STEP1_MANIFEST exists:', STEP1_MANIFEST.exists())
print('RESULT_ROOT:', RESULT_ROOT)


STEP1_META exists: True
STEP1_MANIFEST exists: True
RESULT_ROOT: /content/tennis_data/results_step2


In [4]:
# =============================
# Optional sparse download for selected actions
# =============================

def run_cmd(cmd, cwd=None):
    print('RUN:', ' '.join(cmd))
    out = subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=True)
    if out.stdout.strip():
        print(out.stdout.strip()[:1200])
    if out.stderr.strip():
        print(out.stderr.strip()[:1200])

if USE_GITHUB_DOWNLOAD:
    tmp_repo = Path('/content/thetis_repo_step2')
    if tmp_repo.exists():
        shutil.rmtree(tmp_repo)

    run_cmd(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
             'https://github.com/THETIS-dataset/dataset.git', str(tmp_repo)])

    run_cmd(['git', 'sparse-checkout', 'set'] + [f'VIDEO_RGB/{a}' for a in ACTION_FOLDERS], cwd=str(tmp_repo))

    for action in ACTION_FOLDERS:
        src = tmp_repo / 'VIDEO_RGB' / action
        dst = VIDEO_ROOT / action
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

for action in ACTION_FOLDERS:
    n = len(list((VIDEO_ROOT / action).glob('*.avi')))
    print(f'{action}: {n} videos')


RUN: git clone --depth 1 --filter=blob:none --sparse https://github.com/THETIS-dataset/dataset.git /content/thetis_repo_step2
Cloning into '/content/thetis_repo_step2'...
RUN: git sparse-checkout set VIDEO_RGB/backhand VIDEO_RGB/forehand_flat VIDEO_RGB/kick_service VIDEO_RGB/smash
backhand: 165 videos
forehand_flat: 165 videos
kick_service: 165 videos
smash: 165 videos


In [6]:
# Step 1 outputs loader from Google Drive folder
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

STEP1_DIR = Path('/content/drive/MyDrive/step1_outputs')

STEP1_META     = STEP1_DIR / 'metadata_rgb_multi4_with_split.csv'
STEP1_MANIFEST = STEP1_DIR / 'split_manifest.json'

if not STEP1_META.exists():
    raise FileNotFoundError(f'Metadata not found: {STEP1_META}')
if not STEP1_MANIFEST.exists():
    raise FileNotFoundError(f'Manifest not found: {STEP1_MANIFEST}')

print('STEP1_DIR:', STEP1_DIR)
print('STEP1_META exists:', STEP1_META.exists(), STEP1_META)
print('STEP1_MANIFEST exists:', STEP1_MANIFEST.exists(), STEP1_MANIFEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP1_DIR: /content/drive/MyDrive/step1_outputs
STEP1_META exists: True /content/drive/MyDrive/step1_outputs/metadata_rgb_multi4_with_split.csv
STEP1_MANIFEST exists: True /content/drive/MyDrive/step1_outputs/split_manifest.json


In [7]:
# =============================
# Keypoint extraction
# =============================
from ultralytics import YOLO

LANDMARK_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]


def select_best_person(kps_xy_all, kps_conf_all):
    return int(np.argmax(kps_conf_all.mean(axis=1)))


def extract_keypoints_for_video(model, video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    frames = []

    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if MAX_FRAMES is not None and idx >= MAX_FRAMES:
            break

        r = model(frame, verbose=False)[0]
        if r.keypoints is not None and len(r.keypoints.xy) > 0:
            xy_all = r.keypoints.xy.cpu().numpy()
            if r.keypoints.conf is not None:
                conf_all = r.keypoints.conf.cpu().numpy()
            else:
                conf_all = np.ones((xy_all.shape[0], xy_all.shape[1]), dtype=np.float32)

            pidx = select_best_person(xy_all, conf_all)
            xy = xy_all[pidx].astype(np.float32)
            conf = conf_all[pidx].astype(np.float32).reshape(-1, 1)

            low = conf[:, 0] < CONF_THRESHOLD
            xy[low] = 0.0
            conf[low] = 0.0

            arr = np.concatenate([xy, conf], axis=1)
        else:
            arr = np.zeros((17, 3), dtype=np.float32)

        frames.append(arr)
        idx += 1

    cap.release()
    if not frames:
        return None
    return np.stack(frames)


summary_rows = []
if EXTRACT_KEYPOINTS:
    model = YOLO(POSE_MODEL)

    for action in ACTION_FOLDERS:
        in_dir = VIDEO_ROOT / action
        out_dir = KPT_ROOT / action
        out_dir.mkdir(parents=True, exist_ok=True)

        videos = sorted(in_dir.glob('*.avi'))
        for vp in tqdm(videos, desc=f'extract {action}'):
            stem = vp.stem
            out_npy = out_dir / f'{stem}.npy'

            if out_npy.exists():
                arr = np.load(out_npy)
            else:
                arr = extract_keypoints_for_video(model, vp)
                if arr is None:
                    summary_rows.append({'action_folder': action, 'video_name': vp.name, 'status': 'failed'})
                    continue
                np.save(out_npy, arr)

            summary_rows.append({
                'action_folder': action,
                'video_name': vp.name,
                'frame_count': int(arr.shape[0]),
                'mean_conf': float(arr[...,2].mean()),
                'zero_ratio': float((arr[...,2].sum(axis=1)==0).mean()),
                'status': 'ok'
            })

summary_df = pd.DataFrame(summary_rows)
if len(summary_df):
    summary_df.to_csv(RESULT_ROOT / 'extraction_summary.csv', index=False)
    print('Saved extraction summary:', RESULT_ROOT / 'extraction_summary.csv')
summary_df.head()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


extract smash: 100%|██████████| 165/165 [02:39<00:00,  1.03it/s]

Saved extraction summary: /content/tennis_data/results_step2/extraction_summary.csv


,action_folder,video_name,frame_count,mean_conf,zero_ratio,status
0,backhand,p10_backhand_s1.avi,78,0.929690,0.0,ok
1,backhand,p10_backhand_s2.avi,73,0.943095,0.0,ok
2,backhand,p10_backhand_s3.avi,66,0.939247,0.0,ok
3,backhand,p11_backhand_s1.avi,92,0.936996,0.0,ok
4,backhand,p11_backhand_s2.avi,104,0.932686,0.0,ok


In [8]:
print("meta in globals:", "meta" in globals())
print("STEP1_META:", STEP1_META)
print("STEP1_META exists:", STEP1_META.exists())

meta in globals: False
STEP1_META: /content/drive/MyDrive/step1_outputs/metadata_rgb_multi4_with_split.csv
STEP1_META exists: True


In [9]:
meta = pd.read_csv(STEP1_META)
if 'canonical_action' not in meta.columns:
    meta['canonical_action'] = meta['action_folder']
meta['stem'] = meta['video_name'].str.replace('.avi','',regex=False)
meta['keypoint_npy'] = meta.apply(lambda r: str((KPT_ROOT / r['action_folder'] / f"{r['stem']}.npy").resolve()), axis=1)
print("meta loaded:", meta.shape)

meta loaded: (660, 19)


In [10]:
# =============================
# Feature extraction and tensors
# =============================

def extract_features(arr):
    coords = arr[..., :2].astype(np.float32)
    conf = arr[..., 2].astype(np.float32)

    lsh = coords[:,5,:]
    rsh = coords[:,6,:]
    lhip = coords[:,11,:]
    rhip = coords[:,12,:]

    center = (lsh + rsh + lhip + rhip) / 4.0
    scale = np.linalg.norm(lsh - rhip, axis=1, keepdims=True) + 1e-6
    norm = (coords - center[:,None,:]) / scale[:,None,:]

    vel = np.diff(norm, axis=0)
    vel = np.concatenate([vel, vel[-1:]], axis=0) if len(vel) else np.zeros_like(norm)

    acc = np.diff(vel, axis=0)
    acc = np.concatenate([acc, acc[-1:]], axis=0) if len(acc) else np.zeros_like(norm)

    T = norm.shape[0]
    feat = np.concatenate([
        norm.reshape(T,-1),
        vel.reshape(T,-1),
        acc.reshape(T,-1),
        conf,
    ], axis=1)
    return feat.astype(np.float32)


def pad_or_truncate(seq, max_len):
    t, f = seq.shape
    if t >= max_len:
        return seq[:max_len]
    pad = np.zeros((max_len-t, f), dtype=np.float32)
    return np.concatenate([seq, pad], axis=0)

seqs = [extract_features(np.load(p)) for p in meta['keypoint_npy']]
lengths = np.array([s.shape[0] for s in seqs])
MAX_LEN = max(40, int(np.percentile(lengths, 95)))

X = np.stack([pad_or_truncate(s, MAX_LEN) for s in seqs])

action_to_id = {a:i for i,a in enumerate(sorted(meta['action_folder'].unique()))}
meta['action_id'] = meta['action_folder'].map(action_to_id)

y_action = meta['action_id'].values.astype(int)
y_skill = meta['skill_binary'].values.astype(int)

idx_train = np.where(meta['split'].values=='train')[0]
idx_val = np.where(meta['split'].values=='val')[0]
idx_test = np.where(meta['split'].values=='test')[0]

print('X shape:', X.shape)
print('MAX_LEN:', MAX_LEN)
print('action_to_id:', action_to_id)
print('sizes:', len(idx_train), len(idx_val), len(idx_test))


X shape: (660, 103, 119)
MAX_LEN: 103
action_to_id: {'backhand': 0, 'forehand_flat': 1, 'kick_service': 2, 'smash': 3}
sizes: 420 108 132


In [11]:
# =============================
# Metrics helpers + SVM baseline runner
# =============================

def safe_auc(y_true, prob, multiclass=False):
    if len(np.unique(y_true)) < 2:
        return float('nan')
    if multiclass:
        return float(roc_auc_score(y_true, prob, multi_class='ovr'))
    return float(roc_auc_score(y_true, prob))


def eval_metrics(y_true, y_pred, y_prob=None, multiclass=False):
    out = {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
    }
    out['roc_auc'] = safe_auc(y_true, y_prob, multiclass=multiclass) if y_prob is not None else float('nan')
    return out


def run_svm(X, y, seed, multiclass=False):
    Xtr = X[idx_train].reshape(len(idx_train), -1)
    Xvl = X[idx_val].reshape(len(idx_val), -1)
    Xte = X[idx_test].reshape(len(idx_test), -1)

    ytr, yvl, yte = y[idx_train], y[idx_val], y[idx_test]

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xvl = scaler.transform(Xvl)
    Xte = scaler.transform(Xte)

    clf = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=seed)
    clf.fit(Xtr, ytr)

    rows, preds = [], {}
    for split, xs, ys in [('train',Xtr,ytr),('val',Xvl,yvl),('test',Xte,yte)]:
        pred = clf.predict(xs)
        prob = clf.predict_proba(xs)
        score_prob = prob if multiclass else prob[:,1]
        m = eval_metrics(ys, pred, score_prob, multiclass=multiclass)
        rows.append({'split': split, **m})
        preds[split] = (pred, prob, ys)
    return pd.DataFrame(rows), preds

svm_action_df, svm_action_preds = run_svm(X, y_action, SEED, multiclass=True)
svm_action_df.insert(0,'model','svm_action')
svm_action_df.to_csv(RESULT_ROOT / 'metrics_svm_action.csv', index=False)

svm_skill_df, svm_skill_preds = run_svm(X, y_skill, SEED, multiclass=False)
svm_skill_df.insert(0,'model','svm_skill')
svm_skill_df.to_csv(RESULT_ROOT / 'metrics_svm_skill.csv', index=False)

print('SVM action metrics:', svm_action_df, sep='\n')
print('SVM skill metrics:', svm_skill_df, sep='\n')


SVM action metrics:
        model  split  macro_f1  balanced_acc   roc_auc
0  svm_action  train  0.930974      0.930952  0.995843
1  svm_action    val  0.846433      0.842593  0.949017
2  svm_action   test  0.689710      0.689394  0.872207
SVM skill metrics:
       model  split  macro_f1  balanced_acc   roc_auc
0  svm_skill  train  0.970563      0.966667  0.990139
1  svm_skill    val  0.868155      0.866667  0.907986
2  svm_skill   test  0.704121      0.706944  0.761574


In [12]:
# =============================
# LSTM baseline runner
# =============================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, num_classes, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])


def run_lstm(X, y, seed, num_classes, epochs=30):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    Xtr = torch.tensor(X[idx_train], dtype=torch.float32)
    Xvl = torch.tensor(X[idx_val], dtype=torch.float32)
    Xte = torch.tensor(X[idx_test], dtype=torch.float32)
    ytr = torch.tensor(y[idx_train], dtype=torch.long)
    yvl = torch.tensor(y[idx_val], dtype=torch.long)
    yte = torch.tensor(y[idx_test], dtype=torch.long)

    counts = np.bincount(y[idx_train], minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (np.maximum(counts, 1.0) * num_classes)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model = LSTMClassifier(X.shape[-1], num_classes).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(Xtr,ytr), batch_size=16, shuffle=True)

    def infer(Xt, yt):
        model.eval()
        with torch.no_grad():
            logits = model(Xt.to(DEVICE))
            prob = torch.softmax(logits, dim=1).cpu().numpy()
            pred = logits.argmax(dim=1).cpu().numpy()
        y_np = yt.cpu().numpy()
        return pred, prob, y_np

    best_state, best_val_f1 = None, -1
    for ep in range(1, epochs+1):
        model.train()
        for xb,yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        pv, probv, yv = infer(Xvl, yvl)
        val_f1 = f1_score(yv, pv, average='macro')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    rows, preds = [], {}
    for split, Xt, yt in [('train',Xtr,ytr),('val',Xvl,yvl),('test',Xte,yte)]:
        pred, prob, ytrue = infer(Xt, yt)
        score_prob = prob if num_classes>2 else prob[:,1]
        m = eval_metrics(ytrue, pred, score_prob, multiclass=(num_classes>2))
        rows.append({'split': split, **m})
        preds[split] = (pred, prob, ytrue)

    return pd.DataFrame(rows), preds

lstm_action_df, lstm_action_preds = run_lstm(X, y_action, SEED, num_classes=len(action_to_id), epochs=30)
lstm_action_df.insert(0,'model','lstm_action')
lstm_action_df.to_csv(RESULT_ROOT / 'metrics_lstm_action.csv', index=False)

lstm_skill_df, lstm_skill_preds = run_lstm(X, y_skill, SEED, num_classes=2, epochs=30)
lstm_skill_df.insert(0,'model','lstm_skill')
lstm_skill_df.to_csv(RESULT_ROOT / 'metrics_lstm_skill.csv', index=False)

print('LSTM action metrics:', lstm_action_df, sep='\n')
print('LSTM skill metrics:', lstm_skill_df, sep='\n')


LSTM action metrics:
         model  split  macro_f1  balanced_acc   roc_auc
0  lstm_action  train  0.394462      0.421429  0.670945
1  lstm_action    val  0.339025      0.398148  0.640146
2  lstm_action   test  0.354635      0.386364  0.635369
LSTM skill metrics:
        model  split  macro_f1  balanced_acc   roc_auc
0  lstm_skill  train  0.814538      0.818056  0.908148
1  lstm_skill    val  0.806798      0.802083  0.862847
2  lstm_skill   test  0.734833      0.741667  0.808565


In [13]:
# =============================
# Per-action skill metrics on test (professor confound concern)
# =============================
test_meta = meta.iloc[idx_test].reset_index(drop=True)

pred_df = pd.DataFrame({
    'video_name': test_meta['video_name'].values,
    'subject_id': test_meta['subject_id'].values,
    'action_folder': test_meta['action_folder'].values,
    'y_true_skill': y_skill[idx_test],
    'svm_pred_skill': svm_skill_preds['test'][0],
    'svm_prob_expert': svm_skill_preds['test'][1][:,1],
    'lstm_pred_skill': lstm_skill_preds['test'][0],
    'lstm_prob_expert': lstm_skill_preds['test'][1][:,1],
})
pred_df.to_csv(RESULT_ROOT / 'predictions_test_skill_models.csv', index=False)

rows = []
for act in sorted(pred_df['action_folder'].unique()):
    sub = pred_df[pred_df['action_folder']==act]
    y_true = sub['y_true_skill'].values
    for model_name, pred_col, prob_col in [
        ('svm_skill','svm_pred_skill','svm_prob_expert'),
        ('lstm_skill','lstm_pred_skill','lstm_prob_expert'),
    ]:
        y_pred = sub[pred_col].values
        y_prob = sub[prob_col].values
        m = eval_metrics(y_true, y_pred, y_prob, multiclass=False)
        rows.append({'model': model_name, 'action_folder': act, 'n_test': len(sub), **m})

per_action_skill = pd.DataFrame(rows)
per_action_skill.to_csv(RESULT_ROOT / 'per_action_skill_metrics_test.csv', index=False)

print('Saved per-action skill metrics:', RESULT_ROOT / 'per_action_skill_metrics_test.csv')
per_action_skill


Saved per-action skill metrics: /content/tennis_data/results_step2/per_action_skill_metrics_test.csv


,model,action_folder,n_test,macro_f1,balanced_acc,roc_auc
0,svm_skill,backhand,33,0.784716,0.783333,0.911111
1,lstm_skill,backhand,33,0.875940,0.872222,0.970370
2,svm_skill,forehand_flat,33,0.565789,0.566667,0.674074
3,lstm_skill,forehand_flat,33,0.565789,0.566667,0.596296
4,svm_skill,kick_service,33,0.757353,0.766667,0.848148
5,lstm_skill,kick_service,33,0.784716,0.805556,0.933333
6,svm_skill,smash,33,0.694444,0.711111,0.677778
7,lstm_skill,smash,33,0.682692,0.722222,0.848148


In [14]:
# =============================
# Optional multi-seed summary (for stronger report rigor)
# =============================
if RUN_MULTI_SEED:
    seed_rows = []
    for s in MULTI_SEEDS:
        np.random.seed(s)
        random.seed(s)

        svm_skill_s, _ = run_svm(X, y_skill, s, multiclass=False)
        test_svm = svm_skill_s[svm_skill_s['split']=='test'].iloc[0]
        seed_rows.append({'seed': s, 'model': 'svm_skill', 'macro_f1': test_svm['macro_f1'], 'balanced_acc': test_svm['balanced_acc'], 'roc_auc': test_svm['roc_auc']})

        lstm_skill_s, _ = run_lstm(X, y_skill, s, num_classes=2, epochs=20)
        test_lstm = lstm_skill_s[lstm_skill_s['split']=='test'].iloc[0]
        seed_rows.append({'seed': s, 'model': 'lstm_skill', 'macro_f1': test_lstm['macro_f1'], 'balanced_acc': test_lstm['balanced_acc'], 'roc_auc': test_lstm['roc_auc']})

    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(RESULT_ROOT / 'metrics_skill_multiseed_test.csv', index=False)

    summary = seed_df.groupby('model')[['macro_f1','balanced_acc','roc_auc']].agg(['mean','std'])
    summary.to_csv(RESULT_ROOT / 'metrics_skill_multiseed_summary.csv')
    print(summary)
else:
    print('RUN_MULTI_SEED=False -> skipped multi-seed block')


            macro_f1           balanced_acc             roc_auc          
                mean       std         mean       std      mean       std
model                                                                    
lstm_skill  0.665935  0.034652     0.672222  0.036747  0.739738  0.043783
svm_skill   0.704121  0.000000     0.706944  0.000000  0.761728  0.000134


In [15]:
# =============================
# Merge and export final step2 bundle
# =============================
all_metrics = pd.concat([
    pd.read_csv(RESULT_ROOT / 'metrics_svm_action.csv'),
    pd.read_csv(RESULT_ROOT / 'metrics_svm_skill.csv'),
    pd.read_csv(RESULT_ROOT / 'metrics_lstm_action.csv'),
    pd.read_csv(RESULT_ROOT / 'metrics_lstm_skill.csv'),
], ignore_index=True)

all_metrics.to_csv(RESULT_ROOT / 'metrics_all_models.csv', index=False)
summary = all_metrics[all_metrics['split'].isin(['val','test'])].sort_values(['split','macro_f1'], ascending=[True, False])
summary.to_csv(RESULT_ROOT / 'metrics_summary_val_test.csv', index=False)

handoff_note = """# Step 2 Handoff Note

## What this step produced
- SVM and LSTM baselines for action and skill tasks
- Test-set prediction CSV for skill task
- Per-action skill metrics (confound-focused)
- Optional multi-seed summary (if enabled)

## Reporting requirements
1. Mention Step1 subject-disjoint split was reused unchanged.
2. Report class imbalance handling for both models.
3. Include per-action skill table in final report.
4. Do not claim causal interpretation from baseline models.
"""
(RESULT_ROOT / 'step2_handoff_note.md').write_text(handoff_note)

print('=== Result files ===')
for p in sorted(RESULT_ROOT.glob('*')):
    print('-', p.name)

import shutil
from google.colab import files
zip_path = '/content/step2_outputs.zip'
shutil.make_archive('/content/step2_outputs', 'zip', RESULT_ROOT)
print('Created:', zip_path)
files.download(zip_path)


=== Result files ===
- extraction_summary.csv
- metrics_all_models.csv
- metrics_lstm_action.csv
- metrics_lstm_skill.csv
- metrics_skill_multiseed_summary.csv
- metrics_skill_multiseed_test.csv
- metrics_summary_val_test.csv
- metrics_svm_action.csv
- metrics_svm_skill.csv
- per_action_skill_metrics_test.csv
- predictions_test_skill_models.csv
- step2_handoff_note.md
Created: /content/step2_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>